<div style='background:#1a1a2e;color:#eee;padding:18px;border-radius:8px;margin-bottom:12px'>
<h2 style='margin:0'>🍎 EDUCATOR VERSION — Chapter 16: Confidence Intervals</h2>
<p style='margin:6px 0 0'>This notebook contains answer keys, teaching notes, and discussion prompts. Students should use the standard <strong>Chapter_16.ipynb</strong>.</p>
</div>


# Chapter 16: Confidence Intervals in Structural Engineering
## How Certain Are We About This Bridge's Stiffness?

**Learning Objectives:**
- Construct and interpret a confidence interval for a structural measurement
- Explain why wider intervals mean more uncertainty — and more required safety margin
- Describe how bridge load testing uses confidence intervals to set safe carrying capacities
- Connect the width of a confidence interval to sample size and measurement variability


<center>
<img src='https://upload.wikimedia.org/wikipedia/commons/thumb/9/96/I-35W_Saint_Anthony_Falls_Bridge.jpg/1280px-I-35W_Saint_Anthony_Falls_Bridge.jpg' width='700' />

<em>The St. Anthony Falls Bridge, Minneapolis — the replacement for the I-35W bridge that collapsed in 2007. Before opening in September 2008, MnDOT placed 2 million pounds of precast concrete barriers on the bridge and measured deflections at 36 locations, comparing them to design predictions. This is proof load testing: each measurement is one EI estimate, and the set of measurements forms a confidence interval for the bridge's actual stiffness. (Wikimedia Commons — public domain.)</em>
</center>

---


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Context

**Curriculum connections:**
- *Statistics class*: Confidence intervals; margin of error; effect of sample size and confidence level on interval width; interpretation of 'confidence'
- *Physics class*: Measurement uncertainty; how repeated measurements reduce uncertainty; deflection as a measurable physical quantity

**Prerequisites:** Students should understand the normal distribution, standard deviation, and the concept of a sampling distribution.

**Estimated time:** 50–60 minutes. The coverage probability simulation (Widget 2) takes the longest — let students run it at different confidence levels and observe how often the true value falls inside vs. outside the interval.

**Pedagogical note:** The most persistent confidence interval misconception is that '95% confidence' means 'there is a 95% probability the true value is in this interval.' The correct interpretation — 95% of intervals *constructed by this method* will contain the true value — is subtle. Widget 2 makes this concrete: students watch hundreds of intervals being built and count how many capture the true deflection. This physical demonstration is far more effective than a verbal explanation alone.
</div>


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(16)

# True beam stiffness EI = 120,000 kip·in² (a W18×97 in A572 Gr.50 steel)
# E = 29,000 ksi, I ≈ 4.14 in⁴ for W18×97... actually let's use round numbers
# W18×97: I_x = 1910 in⁴, E = 29000 ksi → EI = 29000 * 1910 = 55,390,000 kip·in²
# For simplicity: represent in units of 1000 kip·ft²
TRUE_EI   = 385.0   # kip·ft² × 1000 (representative value)
MEAS_SD   = 18.0    # measurement noise in same units
print('Bridge load test setup:')
print(f'  True beam stiffness EI: {TRUE_EI} (×10³ kip·ft²)  [hidden from inspector]')
print(f'  Measurement noise σ:    {MEAS_SD} (±{MEAS_SD/TRUE_EI*100:.1f}%)')


## 16.1  What Is a Confidence Interval?

A **confidence interval** is a range of plausible values for an unknown parameter, computed from a sample.

When a structural engineer loads a bridge with known test weights and measures the resulting deflection, they can back-calculate the bridge's **flexural stiffness EI** using Hibbeler's beam deflection formulas (Chapter 8). For a simply supported beam with a midspan point load P:

$$EI = \frac{PL^3}{48\,\delta}$$

But every deflection measurement has noise — sensor error, temperature effects, traffic vibration. Each test gives a slightly different EI estimate. The **confidence interval** captures that uncertainty:

$$\bar{x} \pm t^* \cdot \frac{s}{\sqrt{n}}$$

Where:
- $\bar{x}$ = sample mean of EI estimates
- $s$ = sample standard deviation of EI estimates  
- $n$ = number of load tests
- $t^*$ = critical value from the t-distribution (depends on confidence level and n)

**Why this matters for safety:** If the confidence interval for EI is wide, the engineer cannot be sure whether the bridge is stiffer or more flexible than expected. A more flexible bridge deflects more — meaning stress is higher — meaning the safe load rating must be lower. **Uncertainty directly reduces the load the bridge is allowed to carry.**


## 🔬 Interactive Experiment 1: Building a Confidence Interval From Load Tests

You are an engineer performing a proof load test on a highway bridge. Each test — lowering a known weight onto the bridge and measuring midspan deflection — gives you one EI estimate. The true EI is unknown to you (but the simulation knows it).

Use the slider to increase the number of tests. Watch how the confidence interval narrows — and how the interval captures the true value.


In [ ]:
def _ci_plot(n_tests, confidence):
    sample = np.random.normal(TRUE_EI, MEAS_SD, n_tests)
    xbar = np.mean(sample)
    s    = np.std(sample, ddof=1)
    alpha = 1 - confidence/100
    t_star = stats.t.ppf(1 - alpha/2, df=n_tests-1)
    margin = t_star * s / np.sqrt(n_tests)
    ci_lo, ci_hi = xbar - margin, xbar + margin
    captures = ci_lo <= TRUE_EI <= ci_hi

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: individual test results
    ax1.scatter(range(1, n_tests+1), sample, color='steelblue', alpha=0.6, s=25, zorder=3)
    ax1.axhline(xbar,    color='steelblue', lw=2,   ls='-',  label=f'Sample mean: {xbar:.1f}')
    ax1.axhline(TRUE_EI, color='black',     lw=2,   ls='--', label=f'True EI: {TRUE_EI}')
    ax1.fill_between([0, n_tests+1], [ci_lo]*2, [ci_hi]*2, alpha=0.2,
        color='steelblue', label=f'{confidence}% CI: [{ci_lo:.1f}, {ci_hi:.1f}]')
    ax1.set_xlabel('Load Test Number', fontsize=12)
    ax1.set_ylabel('EI Estimate (×10³ kip·ft²)', fontsize=12)
    ax1.set_title(f'{n_tests} Load Tests — {confidence}% Confidence Interval', fontsize=13)
    ax1.legend(fontsize=9)
    color = 'lightgreen' if captures else 'mistyrose'
    ax1.annotate(f'CI width: {2*margin:.1f}\n{"✅ Captures true EI" if captures else "❌ Misses true EI"}',
        xy=(0.03, 0.05), xycoords='axes fraction', fontsize=10,
        bbox=dict(boxstyle='round', facecolor=color, alpha=0.9))

    # Right: margin of error vs sample size
    ns = range(2, 101)
    margins = [stats.t.ppf(1-alpha/2, df=k-1) * MEAS_SD / np.sqrt(k) for k in ns]
    ax2.plot(list(ns), margins, 'steelblue', lw=2.5)
    ax2.axvline(n_tests, color='firebrick', lw=2, ls='--', label=f'Your n = {n_tests}')
    ax2.axhline(margin, color='firebrick', lw=1.5, ls=':', label=f'Your margin = ±{margin:.1f}')
    ax2.set_xlabel('Number of Load Tests (n)', fontsize=12)
    ax2.set_ylabel(f'Margin of Error (×10³ kip·ft²)', fontsize=12)
    ax2.set_title(f'How Margin of Error Shrinks With More Tests\n({confidence}% confidence)', fontsize=13)
    ax2.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

w_n  = widgets.IntSlider(value=5, min=2, max=80, step=1,
    description='Load tests (n):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_cl = widgets.Dropdown(options=[90, 95, 99], value=95,
    description='Confidence level %:', style={'description_width':'initial'},
    layout=widgets.Layout(width='300px'))
out1 = widgets.interactive_output(_ci_plot, {'n_tests': w_n, 'confidence': w_cl})
display(widgets.VBox([w_n, w_cl, out1]))


### 💬 Stop and Think

1. With n = 3 tests, how wide is your confidence interval? What does that width mean for the engineer deciding how much load the bridge can safely carry?
2. How many tests does it take before the margin of error drops below 10 units? Below 5 units?
3. Why does switching from 90% to 99% confidence make the interval *wider* — even with the same data?


<div style='background:#d4edda;padding:15px;border-left:5px solid #28a745;margin:10px 0'>

### ✅ Answer Key — Stop and Think

**1. Width of CI at n = 3 and what it means for load rating:**
With only 3 measurements, the margin of error is large — typically ±15–25 units depending on the simulated variability. For a load rating decision, a wide interval means the engineer cannot be confident the bridge's true stiffness is close to the measured mean. The conservative response is to set a lower safe load rating — essentially penalizing uncertainty. The wider the CI, the lower the allowable load.

**2. When does margin of error drop below 10 units? Below 5 units?**
This depends on the standard deviation in the widget (σ ≈ 20 units by default). Using ME = z* × σ/√n:
- ME < 10: √n > z* × 20/10 = 2z*; for 95% CI (z*=1.96), n > 15.4 → **n ≥ 16**
- ME < 5: n > 4 × 15.4 = 61.6 → **n ≥ 62**
Halving the margin of error requires *four times* as many measurements — a key insight about diminishing returns in sampling.

**3. Why does 99% CI produce a wider interval than 95% CI with the same data?**
To be more confident of capturing the true value, you must cast a wider net. The z* multiplier increases from 1.96 (95%) to 2.576 (99%). More confidence = wider interval — you cannot have both high confidence and high precision without more data.

**Common misconception:** 'A 95% CI means there's a 5% chance my interval is wrong.' Partially correct — but the probabilistic statement is about the *method*, not about this specific interval. Once you've computed an interval, the true value either is or isn't in it. The 95% refers to the long-run performance of the method.
</div>


---
## ⚠️  Real-World Case: Proof Load Testing the St. Anthony Falls Bridge (2008)

When the I-35W bridge collapsed in Minneapolis on August 1, 2007 (see Chapter 10), the replacement — the St. Anthony Falls Bridge — was designed and built in just 13 months. Before it opened to traffic in September 2008, the Minnesota Department of Transportation (MnDOT) conducted a full-scale **proof load test**.

**What they did:**

Engineers placed approximately 2 million pounds of precast concrete barriers at 18 positions across the bridge deck. At each loading configuration, they measured deflections at 36 sensor locations using high-precision surveying equipment. Each measurement produced one estimate of the bridge's actual flexural stiffness EI. The engineers then compared those EI estimates to what the finite element model had predicted.

**The confidence interval connection:**

Across the 36 sensor locations and multiple loading configurations, the EI estimates scattered around a mean — with variability from sensor noise, thermal effects, and model simplifications. The engineers computed a confidence interval for the actual bridge stiffness:

- A **narrow CI** would mean the measurements agreed closely, the bridge was behaving as designed, and the load rating could be set confidently
- A **wide CI** would indicate unexpected variability — a reason to investigate further before certifying the bridge for full traffic loading

The St. Anthony Falls Bridge results showed close agreement with design predictions. The CI was narrow, and the bridge opened on schedule.

> *Every number the engineers reported — the estimated stiffness, the allowable load — came with a range of uncertainty. That range is the confidence interval. Ignoring it would mean pretending the measurements were exact. They never are.*


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Note — St. Anthony Falls Bridge Proof Load Test

The 2008 MnDOT proof load test is a real-world confidence interval problem: engineers measured deflections at 36 sensor locations under 2 million pounds of load, then compared measured results to finite element model predictions.

**Connecting to the chapter:**
- Each deflection measurement is one draw from the population of possible measurements (sensor error, load positioning variability, etc.)
- The measured mean deflection at each location gives a point estimate
- The spread across repeated load positions and sensor readings gives the uncertainty
- The confidence interval around each measurement determines whether the measured deflection is statistically consistent with the model prediction

**Why this test mattered:** The I-35W collapse (Chapter 10) had shaken public confidence in bridge inspection. MnDOT needed to demonstrate — with data — that the replacement bridge was behaving as modeled. Good agreement between measured and predicted deflections within confidence bounds was the evidence the public needed.

**Teaching connection:** This chapter closes the loop on Chapters 10–16. Chapter 10 showed how observational studies can be confounded. Chapters 11–15 built probability tools. Chapter 16 shows how confidence intervals let engineers make *formal statements of uncertainty* about physical measurements — turning data into defensible engineering decisions.
</div>


## 🔬 Interactive Experiment 2: Coverage Probability

A **95% confidence interval** means: if you repeated your sampling procedure many times, about 95% of the resulting intervals would capture the true parameter.

The simulation below runs 100 repeated load-test campaigns, each producing a confidence interval. Count how many of the 100 intervals capture the true EI — it should be close to your chosen confidence level.


In [ ]:
def _coverage_plot(n_tests, confidence, n_campaigns):
    alpha = 1 - confidence/100
    intervals = []
    for _ in range(n_campaigns):
        s = np.random.normal(TRUE_EI, MEAS_SD, n_tests)
        xb, sd = np.mean(s), np.std(s, ddof=1)
        t_star = stats.t.ppf(1-alpha/2, df=n_tests-1)
        m = t_star * sd / np.sqrt(n_tests)
        intervals.append((xb-m, xb+m, xb-m <= TRUE_EI <= xb+m))

    covered = sum(1 for _,_,c in intervals if c)

    fig, ax = plt.subplots(figsize=(12, 7))
    for i, (lo, hi, cap) in enumerate(intervals):
        color = 'steelblue' if cap else 'firebrick'
        ax.plot([lo, hi], [i, i], color=color, lw=1.5, alpha=0.7)
        ax.plot((lo+hi)/2, i, 'o', color=color, ms=3)
    ax.axvline(TRUE_EI, color='black', lw=2.5, ls='--', label=f'True EI = {TRUE_EI}')
    ax.set_xlabel('EI Estimate (×10³ kip·ft²)', fontsize=12)
    ax.set_ylabel('Campaign Number', fontsize=12)
    ax.set_title(
        f'{covered}/{n_campaigns} intervals capture true EI  '
        f'({covered/n_campaigns*100:.0f}% vs. {confidence}% target)',
        fontsize=13)
    ax.legend(fontsize=10)
    blue_p = mpatches.Patch(color='steelblue', label=f'Captures true EI ({covered})')
    red_p  = mpatches.Patch(color='firebrick', label=f'Misses true EI ({n_campaigns-covered})')
    ax.legend(handles=[blue_p, red_p], fontsize=10)
    plt.tight_layout()
    plt.show()

# need mpatches in scope
import matplotlib.patches as mpatches

w_n  = widgets.IntSlider(value=8, min=2, max=40, step=1,
    description='Tests per campaign (n):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_cl = widgets.Dropdown(options=[90, 95, 99], value=95,
    description='Confidence level %:', style={'description_width':'initial'},
    layout=widgets.Layout(width='300px'))
w_nc = widgets.IntSlider(value=100, min=20, max=200, step=10,
    description='Campaigns:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out2 = widgets.interactive_output(_coverage_plot,
    {'n_tests':w_n,'confidence':w_cl,'n_campaigns':w_nc})
display(widgets.VBox([w_n, w_cl, w_nc, out2]))


---
## 📋  Chapter 16 Review

| Term | Meaning |
|------|--------|
| **Confidence interval** | Range of plausible values for a parameter, from sample data |
| **Confidence level** | Percentage of intervals (from repeated sampling) that would capture the true parameter |
| **Margin of error** | Half-width of the CI: t* × s / √n |
| **t-distribution** | Used instead of normal distribution when σ is unknown and n is small |
| **Coverage probability** | Fraction of CIs (across many samples) that actually contain the true value |

**The Big Idea:** When a structural engineer reports a load rating for a bridge, that rating is not a single certain number — it is the lower bound of a confidence interval for the bridge's true capacity. The wider the interval (more uncertainty), the lower the safe load rating must be. Confidence intervals translate statistical uncertainty into practical engineering decisions: how much load can we safely allow on this structure, given what we have measured and how precisely we measured it?


<div style='background:#d1ecf1;padding:15px;border-left:5px solid #17a2b8;margin:10px 0'>

### 💬 Discussion Prompts

**1. Coverage probability (warm-up):** Run Widget 2 at 95% confidence with 200 simulations. Approximately how many intervals miss the true value? *(~10 out of 200.) Run it again — do you get the same intervals? No — each is a fresh sample. This is why 'confidence' refers to the method, not any single interval.)*

**2. Type I and Type II errors in load rating (pair discussion):** Suppose a bridge's confidence interval just barely includes a deflection that would allow a 10-ton load rating. If you grant the 10-ton rating and the bridge is actually weaker (the true value falls outside the interval), what happens? If you deny the 10-ton rating and the bridge is actually fine, what happens? *(Connects CI decisions to Type I/II error framing: one risk is structural; the other is economic.)*

**3. Sample size and budget (small group):** From the Stop and Think answer, cutting the margin of error in half requires 4× as many sensor readings. If each reading costs $500 in equipment time, how much does it cost to go from ±10 to ±5 units of precision? Is that precision worth the cost for a routine inspection? For approving a new bridge?

**4. Extension (homework):** Find a news article that reports a poll result with a margin of error. Write one paragraph explaining: (a) what the margin of error means, (b) what confidence level was likely used, and (c) what sample size would be needed to cut the margin of error in half.
</div>
